# 01: Build your first circuit, one gate at a time

You will start with two qubits in `00`, add one gate, inspect the change, and then create a Bell state.
All the computation is written here; you do not need to open a helper script.
Complete [Setup](../SETUP.md), then run one cell at a time with **Shift+Enter**.

**Learning goal:** build, draw and run a circuit, then read amplitudes and probabilities.

**Prerequisites:** basic Python variables and function calls; no PyTorch training knowledge or cloud account. If quantum states are new, follow this lesson before 02.

Choose a route in the [workshop guide](../README.md).


## 1. Import the framework

`fq` is just a short name for FlagQuantum. Check the path below if you are unsure which installation your notebook is using.

In [ ]:
import os
import subprocess
from pathlib import Path

import flagquantum as fq

package_path = Path(fq.__file__).resolve()
source_commit = os.getenv("FLAGQUANTUM_SOURCE_COMMIT")
if not source_commit:
    try:
        source_commit = subprocess.run(
            ["git", "-C", str(package_path.parent), "rev-parse", "HEAD"],
            check=True,
            capture_output=True,
            text=True,
            timeout=5,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        source_commit = "not recorded by this deployment"
print("FlagQuantum version:", fq.__version__)
print("FlagQuantum source:", package_path)
print("FlagQuantum commit:", source_commit)
print("QUAFU_API_KEY configured:", bool(os.getenv("QUAFU_API_KEY")))
print("QUAFU_TASK_SERVER_URL configured:", bool(os.getenv("QUAFU_TASK_SERVER_URL")))


## 2. Create two qubits

A new circuit starts with every qubit in state `0`. Creating the circuit describes the program; `fq.run` executes it.

**Before running:** which one of `00`, `01`, `10`, and `11` should have probability one?

Each horizontal line in the circuit diagram is a qubit; gates execute from left to right. Vertical wire order is not the bit-string display order on a cloud platform.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

circuit = fq.Circuit(2)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
print("State amplitudes:", state)


A statevector contains a complex amplitude for each possible bit string. Square the absolute value of each amplitude to get its probability.
The four entries below correspond to `00`, `01`, `10`, and `11`.

**How do you read a bit string?** In FlagQuantum's default full-register order, the bits read `q0, q1` from left to right. Thus `10` means q0=1, q1=0; `01` means q0=0, q1=1. In the statevector index, q0 is the most significant bit (the big-endian convention). For three qubits the order is `q0, q1, q2`.

In [ ]:
labels = ["00", "01", "10", "11"]
probabilities = state.abs().square()
print(dict(zip(labels, probabilities.tolist())))


**Check the order with an asymmetric circuit.** Applying X only to q0 produces `10`, not `01`. We use a separate circuit so the original circuit remains in `00` for the next step.

In [ ]:
order_check = fq.Circuit(2).x(0)
fig, ax = order_check.draw(format="mpl", show_all_qubits=True)
plt.show()
order_state = fq.run(order_check).to_statevector().reshape(-1)
order_probabilities = order_state.abs().square()
assert order_probabilities.tolist() == [0.0, 0.0, 1.0, 0.0]
print("Bit order: q0 q1 (left to right)")
print(dict(zip(labels, order_probabilities.tolist())))


## 3. Add a Hadamard gate

`h(0)` acts on qubit 0 and puts it into a superposition. Qubit 1 still stays at zero.
**Predict first:** which two outcomes will now be possible?

In [ ]:
circuit.h(0)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
print(dict(zip(labels, state.abs().square().tolist())))


## 4. Connect the qubits

`cx(0, 1)` flips qubit 1 when qubit 0 is `1`. With the Hadamard gate already present, this creates a Bell state.
You should now see only `00` and `11`, each with probability 0.5.

Adding a gate changes the circuit object. If you rerun an “add gate” cell, you add that gate again.
To start over, rerun from the cell that creates `circuit`.

**Concept:** amplitudes can be complex; probabilities are their absolute squares. A probability table does not retain relative phases.

**Expected output:** this ideal circuit gives P(00)=P(11)=0.5; the statevector also contains the amplitudes that produced those probabilities.

**Common misconception:** seeing only 00 and 11 does not by itself prove entanglement. An incoherent mixture can have the same distribution; the prepared circuit and additional measurements distinguish it from a Bell state.


In [ ]:
circuit.cx(0, 1)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
probabilities = state.abs().square()
print(dict(zip(labels, probabilities.tolist())))


## 5. Plot and check your answer

This check compares the simulated probabilities with the answer we worked out above.

**Read the output:** `{'00': 0.5, '01': 0.0, '10': 0.0, '11': 0.5}` is the ideal probability table (small numerical differences are normal). Each bar is a probability, not a measured count. The labels read q0 q1, left to right.

**Rerun safely:** rerunning `h(0)` or `cx(0, 1)` appends another gate. Restart from circuit creation for the same experiment; rerunning `fq.run` alone executes the current circuit without appending gates.


In [ ]:
import torch

expected = torch.tensor([0.5, 0.0, 0.0, 0.5])
torch.testing.assert_close(probabilities, expected, atol=1e-6, rtol=1e-6)
plt.bar(labels, probabilities.tolist())
plt.xlabel("Bit string (q0 q1)")
plt.ylabel("Probability")
plt.ylim(0, 1)
plt.show()


## 6. Try your own circuit

Change `use_entangling_gate` below and compare the probabilities. This creates a fresh circuit so you can rerun it safely.
The exact probabilities come from the simulated state. Later, a real device will estimate them using repeated measurements called *shots*.

In [ ]:
use_entangling_gate = False
my_circuit = fq.Circuit(2)
my_circuit.h(0)
if use_entangling_gate:
    my_circuit.cx(0, 1)
fig, ax = my_circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
my_state = fq.run(my_circuit).to_statevector().reshape(-1)
print(dict(zip(labels, my_state.abs().square().tolist())))


## Take it home

This is the complete local Bell example. It requires FlagQuantum and PyTorch,
not the workshop folder, a platform account or a special image.

```python
import flagquantum as fq

bell = fq.Circuit(2).h(0).cx(0, 1)
probabilities = fq.run(bell).to_statevector().reshape(-1).abs().square()
print(probabilities.tolist())  # approximately [0.5, 0, 0, 0.5]
```
